# Locked SSB for Known 4D-STEM Scan Drift

Use this notebook after the forward-model notebook has written `clean0`, `drift0`, and `drift90` exports.

This notebook is intentionally locked-calibration only:

1. Fit SSB calibration on `clean0`.
2. Apply the same `C10/C12/phi12` calibration to `drift0`.
3. Apply the same calibration to `drift90`, testing the `clean0 - 90` and `clean0 + 90` rotation branches.

It does not run free calibration on the drifted datasets.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt

REPO = Path("/home/owner/repos/quantem")
WORKFLOW_DIR = REPO / "notebooks" / "drift" / "dev" / "real"
if str(WORKFLOW_DIR) not in sys.path:
    sys.path.insert(0, str(WORKFLOW_DIR))

from known_scan_drift_runner import (
    KnownScanDriftConfig,
    export_status,
    print_export_status,
    run_locked_ssb,
)

## Parameters

Use the same source H5 and drift vector used in the forward-model notebook.

In [2]:
SOURCE_H5 = Path("/home/owner/ssd/data/dasol/20260415_BTOSTO/BTO_18_master.h5")
DRIFT_TOTAL_PX_DOWN_RIGHT = (0.0, 30.0)

DET_BIN = 2
SAVE_CROP = 400
GPU = 0

VOLTAGE_KV = 300.0
SEMIANGLE_MRAD = 30.0
SCAN_SAMPLING_A = 0.264
SSB_N_TRIALS = 200
SSB_REFINE = "nmead"

RUN_LOCKED_SSB = False

config = KnownScanDriftConfig(
    source_h5=SOURCE_H5,
    drift_total_px_down_right=DRIFT_TOTAL_PX_DOWN_RIGHT,
    det_bin=DET_BIN,
    save_crop=SAVE_CROP,
    gpu=GPU,
    voltage_kv=VOLTAGE_KV,
    semiangle_mrad=SEMIANGLE_MRAD,
    scan_sampling_a=SCAN_SAMPLING_A,
    ssb_n_trials=SSB_N_TRIALS,
    ssb_refine=SSB_REFINE,
)
config.summary()

{'source_h5': '/home/owner/ssd/data/dasol/20260415_BTOSTO/BTO_18_master.h5',
 'dataset_label': 'BTO_18',
 'drift_total_px_down_right': (0.0, 30.0),
 'export_dir': '/home/owner/ssd/data/dasol/20260415_BTOSTO/quantem/drift/real/BTO_18_known_down0_right30_crop400_detbin2_u16',
 'output_dir': '/home/owner/repos/quantem/notebooks/drift/dev/outputs/BTO_18_known_down0_right30'}

## Confirm Inputs

All three masters must exist before running locked SSB.

In [3]:
rows = export_status(config)
try:
    import pandas as pd
    display(pd.DataFrame(rows)[[
        "name",
        "exists",
        "known_drift_total_px_down",
        "known_drift_total_px_right",
        "scan_crop_row_start",
        "scan_crop_row_stop",
        "scan_crop_col_start",
        "scan_crop_col_stop",
        "path",
    ]])
except Exception:
    print_export_status(config)

clean0  missing  /home/owner/ssd/data/dasol/20260415_BTOSTO/quantem/drift/real/BTO_18_known_down0_right30_crop400_detbin2_u16/BTO_18_ground_truth_crop400_detbin2_master.h5
drift0  missing  /home/owner/ssd/data/dasol/20260415_BTOSTO/quantem/drift/real/BTO_18_known_down0_right30_crop400_detbin2_u16/BTO_18_down0_right30_image_0_crop400_detbin2_master.h5
drift90 missing  /home/owner/ssd/data/dasol/20260415_BTOSTO/quantem/drift/real/BTO_18_known_down0_right30_crop400_detbin2_u16/BTO_18_down0_right30_image_1_crop400_detbin2_master.h5


## Run Locked SSB

Set `RUN_LOCKED_SSB = True` in the parameter cell. The only optimized calibration is the clean0 reference; drifted datasets keep that calibration locked.

In [4]:
if RUN_LOCKED_SSB:
    summary = run_locked_ssb(config)
else:
    print("RUN_LOCKED_SSB is False; not running SSB.")

RUN_LOCKED_SSB is False; not running SSB.


## Inspect Locked Result

In [5]:
if config.locked_figure_path.exists():
    fig, ax = plt.subplots(figsize=(13, 7))
    ax.imshow(plt.imread(config.locked_figure_path))
    ax.set_axis_off()
    ax.set_title(config.locked_figure_path.name)
else:
    print(f"No locked SSB figure yet: {config.locked_figure_path}")

if config.locked_summary_json.exists():
    with config.locked_summary_json.open() as f:
        locked_summary = json.load(f)
    print(json.dumps(locked_summary["locked"], indent=2))
else:
    print(f"No locked SSB summary yet: {config.locked_summary_json}")

No locked SSB figure yet: /home/owner/repos/quantem/notebooks/drift/dev/outputs/BTO_18_known_down0_right30/BTO_18_known_down0_right30_locked_ssb.png
No locked SSB summary yet: /home/owner/repos/quantem/notebooks/drift/dev/outputs/BTO_18_known_down0_right30/locked_ssb/locked_from_clean0_summary.json


## Outputs

In [6]:
print("Locked SSB figure:", config.locked_figure_path)
print("Locked SSB summary:", config.locked_summary_json)

Locked SSB figure: /home/owner/repos/quantem/notebooks/drift/dev/outputs/BTO_18_known_down0_right30/BTO_18_known_down0_right30_locked_ssb.png
Locked SSB summary: /home/owner/repos/quantem/notebooks/drift/dev/outputs/BTO_18_known_down0_right30/locked_ssb/locked_from_clean0_summary.json
